In [131]:
import pandas as pd

### 진행해야할 사항
- 컬럼 정리 -> 필요없는 컬럼 우선 제거 "대여구분코드", "성별", "운동량", "탄소량"
- "대여일자" 컬럼 "연도", "월", "일" 컬럼으로 나눌 필요가 있음
- "대여소명" 컬럼 "대여소번호" 와 "대여소"로 나눌 필요 있음
- 각 항목에 필요한 데이터 파일을 분리할 필요가 있음

#### 데이터 항목
- 2025년 월별 이용 추이
- 인기 대여소 6곳
- 연령대별 이용 비율
- 당장은 이렇게 세 개의 데이터

In [118]:
df1 = pd.read_csv('data/time/서울특별시 공공자전거 이용정보(시간대별)_202508.csv')
df1.head()

,대여일자,대여시간,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
0,2025-07-01,0,1293,1293. 석촌호수 서호 사거리,정기권,NaN,~10대,1,15.83,0.14,615.10,5
1,2025-07-01,0,739,739. 신월사거리,정기권,NaN,20대,1,9.81,0.09,381.15,4
2,2025-07-01,0,505,505. 자양사거리 광진아크로텔 앞,정기권,NaN,20대,2,31.36,0.33,1393.80,8
3,2025-07-01,0,1529,1529. 미아동 한국전력공사,정기권,NaN,20대,1,79.58,0.72,3091.62,16
4,2025-07-01,0,1650,1650. 중계근린공원내,정기권,NaN,20대,2,76.78,0.69,2982.88,38


In [101]:
df_top.to_csv('data/월별_대여소별_공공자전거_이용건수_전체.csv', index=False, encoding='utf-8-sig')

print('데이터 저장 완료!')

데이터 저장 완료!


In [132]:
for month in range(1, 7):
    
    # 파일 경로
    file = f'data/time/서울특별시 공공자전거 이용정보(시간대별)_2025{month:02d}.csv'
    
    # CSV 불러오기
    df = pd.read_csv(file)

    # 대여소번호와 대여소 분리
    df[['대여소번호', '대여소']] = df['대여소명'].str.extract(
        r'^(\d+)\.\s*(.*)$'
    )

    # 대여소번호 숫자 변환
    df['대여소번호'] = pd.to_numeric(
        df['대여소번호'],
        errors='coerce'
    )

    # 대여소 이름 공백 제거
    df['대여소'] = df['대여소'].str.strip()

    # 날짜 변환
    df['대여일자'] = pd.to_datetime(
        df['대여일자'],
        errors='coerce'
    )

    # 연도 / 월 / 일 분리
    df['연도'] = df['대여일자'].dt.year
    df['월'] = df['대여일자'].dt.month
    df['일'] = df['대여일자'].dt.day

    # 원하는 컬럼 순서
    cols = ['연도', '월', '일'] + [
        col for col in df.columns
        if col not in ['연도', '월', '일']
    ]

    df = df[cols]

    # 불필요한 컬럼 삭제
    df = df.drop(
        ['대여소명', '대여일자', '대여구분코드', '성별', '운동량', '탄소량'],
        axis=1
    )

    # 대여소번호 정수형 변환
    df['대여소번호'] = df['대여소번호'].astype('Int64')

    # 대여소번호 결측치 제거
    df = df.dropna(subset=['대여소번호'])

    # 컬럼 이름 변경
    df = df.rename(columns={'이동거리': '이동거리(M)'})

    # # 대여소별 이용건수 합계
    # df = (
    #     df.groupby(
    #         ['연도', '월', '대여소번호', '대여소'],
    #         as_index=False
    #     )['이용건수']
    #     .sum()
    #     .reset_index(drop=True)
    # )

    # df_202501, df_202502 ... 형태로 저장
    globals()[f'df_2025{month:02d}'] = df

print('처리완료!')

처리완료!


In [133]:
df_202501.head(10)

,연도,월,일,대여시간,대여소번호,연령대코드,이용건수,이동거리(M),대여소
0,2025,1,1,0,3684,~10대,1,4730.80,고덕자이(105동 앞)
1,2025,1,1,0,1044,20대,1,1314.19,굽은다리역
2,2025,1,1,0,1153,20대,1,992.04,"발산역 1번, 9번 인근 대여소"
3,2025,1,1,0,1260,20대,1,3187.35,방이동 한양3차아파트 옆
4,2025,1,1,0,1845,20대,1,2090.00,롯데캐슬골드파크1차 서문
5,2025,1,1,0,1338,20대,1,2840.00,용문2교 옆
6,2025,1,1,0,1725,20대,1,700.00,창1동주민센터
7,2025,1,1,0,1452,20대,1,1436.47,겸재교 진입부
8,2025,1,1,0,1278,20대,1,2273.58,송파구청 교차로
9,2025,1,1,0,1543,20대,1,1370.00,수유1동 주민센터


In [138]:
df7 = pd.read_csv('data/time/서울특별시 공공자전거 이용정보(시간대별)_202507.csv')
df7.tail()

,대여일자,대여시간,대여소번호,대여소명,대여구분코드,성별,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
3222849,2025-07-31,23,1637,1637. KT 전화국 버스정류장 옆,정기권,M,기타,1,65.31,0.49,2114.27,12
3222850,2025-07-31,23,1901,1901. 신도림동주민센터 앞,정기권,M,기타,1,45.71,0.46,1990.00,33
3222851,2025-07-31,23,2102,2102. 봉림교 교통섬,정기권,M,기타,1,44.81,0.38,1616.58,107
3222852,2025-07-31,23,2048,2048. 삼일초등학교 인근,정기권,M,기타,1,30.09,0.30,1310.00,7
3222853,2025-07-31,23,2244,2244. 말죽거리공원 사거리,정기권,M,기타,1,19.25,0.17,715.00,5


In [134]:
for month in range(7, 13):
    
    # 파일 경로
    file = f'data/time/서울특별시 공공자전거 이용정보(시간대별)_2025{month:02d}.csv'
    
    # CSV 불러오기
    df1 = pd.read_csv(file)

    # 대여소번호와 대여소 분리
    df1[['대여소번호', '대여소']] = df1['대여소명'].str.extract(
        r'^(\d+)\.\s*(.*)$'
    )

    # 대여소번호 숫자 변환
    df1['대여소번호'] = pd.to_numeric(
        df['대여소번호'],
        errors='coerce'
    )

    # 대여소 이름 공백 제거
    df1['대여소'] = df1['대여소'].str.strip()

    # 날짜 변환
    df1['대여일자'] = pd.to_datetime(
        df1['대여일자'],
        errors='coerce'
    )

    # 연도 / 월 / 일 분리
    df1['연도'] = df1['대여일자'].dt.year
    df1['월'] = df1['대여일자'].dt.month
    df1['일'] = df1['대여일자'].dt.day

    # 원하는 컬럼 순서
    cols = ['연도', '월', '일'] + [
        col for col in df1.columns
        if col not in ['연도', '월', '일']
    ]

    df1 = df1[cols]

    # 불필요한 컬럼 삭제
    df1 = df1.drop(
        ['대여소명', '대여일자', '대여구분코드', '성별', '운동량', '탄소량', '이용시간(분)'],
        axis=1
    )

    # 대여소번호 정수형 변환
    df1['대여소번호'] = df1['대여소번호'].astype('Int64')

    # 대여소번호 결측치 제거
    df1 = df1.dropna(subset=['대여소번호'])

    # # 대여소별 이용건수 합계
    # df = (
    #     df.groupby(
    #         ['연도', '월', '대여소번호', '대여소'],
    #         as_index=False
    #     )['이용건수']
    #     .sum()
    #     .reset_index(drop=True)
    # )

    # df_202501, df_202502 ... 형태로 저장
    globals()[f'df_2025{month:02d}'] = df1

print('처리완료!')

처리완료!


In [140]:
df_202501.tail()

,연도,월,일,대여시간,대여소번호,연령대코드,이용건수,이동거리(M),대여소
1610870,2025,1,31,23,3500,60대,1,3210.00,군자역2번출구
1610871,2025,1,31,23,153,기타,1,3517.82,성산2교 사거리
1610872,2025,1,31,23,4317,기타,1,877.00,양재근린공원
1610873,2025,1,31,23,139,기타,1,2777.12,연세대 정문 건너편
1610874,2025,1,31,23,414,기타,1,330.00,상암동주민센터 옆


In [136]:
for month in range(1, 13):
    df = globals()[f'df_2025{month:02d}']
    
    df.to_csv(
        f'data/scaled_publicBike_data/월별_대여소별_공공자전거_이용건수_2025{month:02d}.csv',
        index=False,
        encoding='utf-8-sig'
    )

print('데이터 저장 완료!')

데이터 저장 완료!


In [141]:
print(df_202501.columns.tolist())

['연도', '월', '일', '대여시간', '대여소번호', '연령대코드', '이용건수', '이동거리(M)', '대여소']
